# RSNA Knee Abnormality Detection - Colab baseline

Versión compatible con **Google Colab** del baseline original de Kaggle. Mantiene el mismo flujo: extracción de características DICOM + TF-IDF del reporte + 12 regresiones logísticas para generar `submission.csv`.

> Ejecuta las celdas en orden. La GPU es opcional (el pipeline es principalmente CPU-bound) y la notebook funciona también en CPU.


In [1]:
from pathlib import Path
import importlib
import importlib.metadata
import subprocess
import sys

MIN_VERSIONS = {
    "joblib": "1.3",
    "numpy": "1.24",
    "pandas": "2.0",
    "pydicom": "2.4",
    "scikit-learn": "1.3",
}
IMPORT_NAMES = {
    "scikit-learn": "sklearn",
}

def _version_tuple(value):
    parts = []
    for token in str(value).split('.'):
        number = ''.join(ch for ch in token if ch.isdigit())
        if not number:
            break
        parts.append(int(number))
    return tuple(parts)


def ensure_dependencies(min_versions):
    to_install = []
    for package, minimum in min_versions.items():
        module_name = IMPORT_NAMES.get(package, package)
        if importlib.util.find_spec(module_name) is None:
            to_install.append(f"{package}>={minimum}")
            continue
        try:
            current = importlib.metadata.version(package)
            if _version_tuple(current) < _version_tuple(minimum):
                to_install.append(f"{package}>={minimum}")
        except importlib.metadata.PackageNotFoundError:
            to_install.append(f"{package}>={minimum}")
    if to_install:
        print("Installing/Updating:", ", ".join(to_install))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
    else:
        print("All required packages already available.")


def report_runtime():
    print("Python:", sys.version.split()[0])
    print("Executable:", sys.executable)
    has_gpu = subprocess.run(["bash", "-lc", "command -v nvidia-smi >/dev/null && nvidia-smi -L"], capture_output=True, text=True)
    if has_gpu.returncode == 0 and has_gpu.stdout.strip():
        print("GPU detected:", has_gpu.stdout.strip())
    else:
        print("GPU not detected. Continuing on CPU (supported).")


ensure_dependencies(MIN_VERSIONS)
report_runtime()


All required packages already available.
Python: 3.13.15
Executable: /usr/bin/python3
GPU detected: GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-5922c00d-f655-db69-59be-965188f6b46f)


In [2]:
from pathlib import Path
import subprocess

try:
    from google.colab import drive  # type: ignore
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

GITHUB_REPO_URL = "https://github.com/KarlaRoChi/RSNA-Knee-Abnormality-Detection.git"
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIRNAME = "RSNA-Knee-Abnormality-Detection"
CLONE_REPO_IF_MISSING = True

REPO_ROOT = WORKSPACE_ROOT / REPO_DIRNAME
if CLONE_REPO_IF_MISSING and IN_COLAB:
    if (REPO_ROOT / ".git").exists():
        print("Using existing repository clone:", REPO_ROOT)
    elif REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        print("Directory exists and is not empty, skipping clone:", REPO_ROOT)
    else:
        # Intentar obtener el GITHUB_TOKEN de los secretos de Colab
        try:
            github_token = userdata.get('GITHUB_TOKEN')
            if github_token:
                # Reemplazar https:// con https://<token>@ para autenticar
                clone_url = GITHUB_REPO_URL.replace("https://", f"https://{github_token}@")
                print("Attempting to clone using GITHUB_TOKEN...")
            else:
                clone_url = GITHUB_REPO_URL
                print("No GITHUB_TOKEN secret found. Attempting public clone...")
        except Exception:
            clone_url = GITHUB_REPO_URL
            print("Could not retrieve GITHUB_TOKEN from secrets. Attempting public clone...")

        subprocess.check_call(["git", "clone", clone_url, str(REPO_ROOT)])
        print("Repository cloned to:", REPO_ROOT)
elif not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()

OUTPUT_ROOT = REPO_ROOT / "colab_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Using existing repository clone: /content/RSNA-Knee-Abnormality-Detection
IN_COLAB: True
REPO_ROOT: /content/RSNA-Knee-Abnormality-Detection
OUTPUT_ROOT: /content/RSNA-Knee-Abnormality-Detection/colab_outputs


## Configuración de dataset

El flujo normal en Colab **no requiere Kaggle API**. Usa una de estas opciones:

1. **Google Drive**: monta Drive y apunta `DRIVE_DATASET_DIR`.
2. **Local en Colab**: sube/descomprime el dataset en `/content/...` y usa `LOCAL_DATASET_DIR`.
3. **Opcional Kaggle API**: habilita `USE_KAGGLE_API=True` solo si deseas descargar desde Kaggle dentro de Colab.

Estructura esperada del dataset:

- `train.csv`, `test.csv`, `train_series.csv`, `test_series.csv`
- `train_series/<StudyInstanceUID>/.../*.dcm`
- `test_series/<StudyInstanceUID>/.../*.dcm`


In [3]:
from pathlib import Path
import subprocess
import sys
import os
import shutil
import pandas as pd
import numpy as np
from google.colab import userdata

USE_KAGGLE_API = True
KAGGLE_COMPETITION = "rsna-knee-abnormality-detection"
KAGGLE_DATASET_DIR = "/content/rsna-knee-abnormality-detection"

# Asegurar que el directorio de datos existe
data_root_path = Path(KAGGLE_DATASET_DIR)
data_root_path.mkdir(parents=True, exist_ok=True)

try:
    # Extraer el secreto KAGGLE_API que tiene el token KGAT activo
    token_val = userdata.get('KAGGLE_API')
    kaggle_user = userdata.get('KAGGLE_USERNAME')

    if not kaggle_user:
        kaggle_user = "rochikarla"

    if token_val:
        token_val = token_val.strip()
        os.environ["KAGGLE_USERNAME"] = kaggle_user.strip()
        os.environ["KAGGLE_API_TOKEN"] = token_val

        # Configurar la ruta de tokens de acceso para Kaggle CLI
        kaggle_home = Path.home() / ".kaggle"
        kaggle_home.mkdir(parents=True, exist_ok=True)

        access_token_file = kaggle_home / "access_token"
        with open(access_token_file, "w") as f:
            f.write(token_val)
        access_token_file.chmod(0o600)
        print("🔑 Autenticación mediante Token de Acceso (KGAT) configurada correctamente.")
    else:
        print("❌ Falta el secreto KAGGLE_API en Google Colab.")
except Exception as e:
    print(f"⚠️ Error al configurar credenciales: {e}")

kaggle_success = False
if USE_KAGGLE_API:
    try:
        print("Iniciando descarga del dataset real desde Kaggle...")
        # Descargar el dataset completo de la competencia
        cmd = ["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(data_root_path)]
        res = subprocess.run(cmd, capture_output=True, text=True)

        if res.returncode != 0:
            print("❌ Error retornado por Kaggle CLI:")
            print("STDOUT:", res.stdout)
            print("STDERR:", res.stderr)
        else:
            zip_files = list(data_root_path.glob("*.zip"))
            if zip_files:
                print(f"Descomprimiendo {zip_files[0].name}...")
                subprocess.check_call(["unzip", "-o", "-q", str(zip_files[0]), "-d", str(data_root_path)])
                zip_files[0].unlink()
            kaggle_success = True
            print("🎉 ¡Dataset real descargado y descomprimido correctamente!")
    except Exception as e:
        print(f"❌ Error al descargar de Kaggle: {e}")

DATA_ROOT = data_root_path
print("DATA_ROOT configurado en:", DATA_ROOT)

🔑 Autenticación mediante Token de Acceso (KGAT) configurada correctamente.
Iniciando descarga del dataset real desde Kaggle...


KeyboardInterrupt: 

In [ ]:
import subprocess
from pathlib import Path

# Crear la ruta de datos si no existe
data_root_path = Path("/content/rsna-knee-abnormality-detection")
data_root_path.mkdir(parents=True, exist_ok=True)

print("📥 Descargando únicamente los archivos de metadatos (.csv) de la competencia...")
csv_files = ["train.csv", "test.csv", "train_series.csv", "test_series.csv", "sample_submission.csv"]

for csv_file in csv_files:
    try:
        print(f"Descargando {csv_file}...")
        # Sintaxis correcta del Kaggle CLI para descargar un archivo específico de competencia
        cmd = ["kaggle", "competitions", "download", "-c", "rsna-knee-abnormality-detection", "-f", csv_file, "-p", str(data_root_path)]
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f"❌ No se pudo descargar {csv_file}: {e}")

# Descomprimir si la CLI de Kaggle descargó los archivos comprimidos (.zip)
for zip_file in data_root_path.glob("*.zip"):
    try:
        print(f"Descomprimiendo {zip_file.name}...")
        subprocess.run(["unzip", "-o", "-q", str(zip_file), "-d", str(data_root_path)], check=True)
        zip_file.unlink()
    except Exception as e:
        print(f"❌ No se pudo descomprimir {zip_file.name}: {e}")

print("✅ Proceso de descarga finalizado. Archivos disponibles en:", list(data_root_path.glob("*")))

In [ ]:
import os
import subprocess
from pathlib import Path
from google.colab import userdata

print("=== CONFIGURACIÓN DE TOKEN DE ACCESO KAGGLE (KGAT) ===")

try:
    # Obtenemos dinámicamente el secreto de Colab que acabas de guardar
    token_val = userdata.get('KAGGLE_API')
    kaggle_user = userdata.get('KAGGLE_USERNAME')

    if not kaggle_user:
        kaggle_user = "rochikarla"

    if token_val:
        token_val = token_val.strip()
        # Configurar la variable oficial para tokens KGAT
        os.environ["KAGGLE_USERNAME"] = kaggle_user.strip()
        os.environ["KAGGLE_API_TOKEN"] = token_val

        # Crear la ruta para el archivo de access_token en ~/.kaggle/
        kaggle_home = Path.home() / ".kaggle"
        kaggle_home.mkdir(parents=True, exist_ok=True)

        access_token_file = kaggle_home / "access_token"
        with open(access_token_file, "w") as f:
            f.write(token_val)

        access_token_file.chmod(0o600)
        print("✅ Archivo ~/.kaggle/access_token creado y configurado con permisos 600.")
        print("✅ Variable de entorno KAGGLE_API_TOKEN configurada.")
    else:
        print("❌ No se encontró el secreto KAGGLE_API en la configuración de Colab.")

except Exception as e:
    print(f"⚠️ Error al configurar token: {e}")

print("\n=== PROBANDO CONEXIÓN A KAGGLE CON EL NUEVO TOKEN ===")
try:
    cmd = ["kaggle", "competitions", "files", "-c", "rsna-knee-abnormality-detection"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("🎉 ¡ÉXITO! Conexión autorizada y autenticada con el Token de Acceso de Kaggle.")
        print("Archivos de la competencia detectados:")
        print(result.stdout)
    else:
        print("❌ La autenticación falló con este token:")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
except Exception as e:
    print("❌ Error inesperado al intentar conectar:", e)

In [ ]:
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import pydicom
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
ID = "StudyInstanceUID"
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
print("Configuration ready")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_CSV, TEST_CSV = DATA_ROOT / "train.csv", DATA_ROOT / "test.csv"
TRAIN_SERIES_CSV, TEST_SERIES_CSV = DATA_ROOT / "train_series.csv", DATA_ROOT / "test_series.csv"
TRAIN_ROOT, TEST_ROOT = DATA_ROOT / "train_series", DATA_ROOT / "test_series"

# Verificar estrictamente la existencia de las rutas del dataset oficial de Kaggle
required_paths = [TRAIN_CSV, TEST_CSV, TRAIN_ROOT, TEST_ROOT]
missing_paths = [str(path) for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "❌ No se encontraron los archivos oficiales de la competencia RSNA Knee Abnormality en local.\n"
        f"Rutas faltantes: {', '.join(missing_paths)}\n"
        "Por favor, asegúrate de que la descarga de Kaggle en la celda 'vkHemlhvO3vd' finalice correctamente."
    )

# Cargar dataframes reales de la competencia
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
train_series = pd.read_csv(TRAIN_SERIES_CSV) if TRAIN_SERIES_CSV.exists() else pd.DataFrame()
test_series = pd.read_csv(TEST_SERIES_CSV) if TEST_SERIES_CSV.exists() else pd.DataFrame()

missing_targets = [c for c in TARGETS if c not in train.columns]
if missing_targets:
    raise ValueError(f"Faltan variables objetivo en train.csv: {missing_targets}")

print("✅ Todos los archivos oficiales cargados correctamente.")
print("Ruta del dataset:", DATA_ROOT)
print(f"train shape: {train.shape} | test shape: {test.shape} | series shape: {train_series.shape}")

## Feature extraction

Se muestrean hasta 24 slices por estudio para mantener el tiempo de ejecución. Se calculan cuantiles de intensidad, cantidad de slices y descriptores de `train_series.csv`. En test el reporte está vacío, pero el mismo flujo funciona sin texto.


In [ ]:
def series_features(study_id, frame):
    if frame.empty:
        return np.zeros(6, dtype=np.float32)
    rows = frame[frame[ID].astype(str) == str(study_id)]
    return np.array([
        len(rows),
        rows["Fluid_Sensitive"].sum() if "Fluid_Sensitive" in rows else 0,
        rows["Fat_Suppression"].sum() if "Fat_Suppression" in rows else 0,
        (rows["Anatomical_Plane"] == "Sagittal").sum() if "Anatomical_Plane" in rows else 0,
        (rows["Anatomical_Plane"] == "Coronal").sum() if "Anatomical_Plane" in rows else 0,
        (rows["Anatomical_Plane"] == "Axial").sum() if "Anatomical_Plane" in rows else 0,
    ], dtype=np.float32)

def image_features(study_root, max_slices=24):
    paths = sorted(p for p in Path(study_root).rglob("*") if p.is_file())
    if len(paths) > max_slices:
        paths = [paths[i] for i in np.linspace(0, len(paths) - 1, max_slices, dtype=int)]
    samples = []
    for path in paths:
        try:
            pixels = np.asarray(pydicom.dcmread(path, force=True).pixel_array, dtype=np.float32)
            if pixels.size:
                pixels = np.nan_to_num(pixels)
                lo, hi = np.percentile(pixels, [1, 99])
                if hi > lo:
                    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
                samples.append(pixels.ravel())
        except Exception:
            continue
    if not samples:
        return np.zeros(10, dtype=np.float32)
    values = np.concatenate(samples)
    return np.array([values.mean(), values.std(), *np.percentile(values, [1, 10, 25, 50, 75, 90, 99]), len(samples) / max_slices], dtype=np.float32)

def build_features(frame, image_root, series_frame, has_report, name):
    texts, numeric = [], []
    started = time.perf_counter()
    total = len(frame)
    print(f"Starting {name}: {total} studies", flush=True)
    for n, (_, row) in enumerate(frame.iterrows(), 1):
        study_id = str(row[ID])
        texts.append(str(row.get("Report", "")) if has_report and pd.notna(row.get("Report", "")) else "")
        numeric.append(np.concatenate([image_features(Path(image_root) / study_id), series_features(study_id, series_frame)]))
        if n % 10 == 0 or n == 1 or n == total:
            elapsed = time.perf_counter() - started
            rate = n / elapsed if elapsed else 0
            remaining = (total - n) / rate if rate else 0
            print(f"{name}: {n}/{total} studies | {rate:.2f}/s | ETA {remaining / 60:.1f} min", flush=True)
    print(f"Finished {name} in {(time.perf_counter() - started) / 60:.1f} min", flush=True)
    return texts, np.vstack(numeric)

train_text, train_numeric = build_features(train, TRAIN_ROOT, train_series, True, "train")
test_text, test_numeric = build_features(test, TEST_ROOT, test_series, False, "test")
print(train_numeric.shape, test_numeric.shape)


In [ ]:
vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=1)
text_train = vectorizer.fit_transform(train_text)
text_test = vectorizer.transform(test_text)
scaler = StandardScaler()
numeric_train = scaler.fit_transform(train_numeric)
numeric_test = scaler.transform(test_numeric)
X_train = hstack([text_train, numeric_train]).tocsr()
X_test = hstack([text_test, numeric_test]).tocsr()
print("Design matrices:", X_train.shape, X_test.shape)


In [ ]:
predictions = np.zeros((len(test), len(TARGETS)), dtype=np.float32)
train_predictions = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
models = {}
for j, target in enumerate(TARGETS):
    y = train[target].fillna(0).astype(int).to_numpy()
    if np.unique(y).size < 2:
        predictions[:, j] = y.mean()
        train_predictions[:, j] = y.mean()
        continue
    model = LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED)
    model.fit(X_train, y)
    predictions[:, j] = model.predict_proba(X_test)[:, 1]
    train_predictions[:, j] = model.predict_proba(X_train)[:, 1]
    models[target] = model
auc = [roc_auc_score(train[t], train_predictions[:, j]) for j, t in enumerate(TARGETS) if train[t].nunique() > 1]
print(f"Training macro ROC-AUC: {np.mean(auc):.5f}")


In [ ]:
submission = test[[ID]].copy()
submission[TARGETS] = np.clip(predictions, 0, 1)
submission_path = OUTPUT_ROOT / "submission.csv"
model_path = OUTPUT_ROOT / "rsna_knee_baseline.joblib"

submission.to_csv(submission_path, index=False)
assert list(submission.columns) == [ID] + TARGETS
assert submission[TARGETS].notna().all().all()
joblib.dump({"targets": TARGETS, "vectorizer": vectorizer, "scaler": scaler, "models": models}, model_path)

print("Submission saved:", submission_path)
print("Model bundle saved:", model_path)
display(submission.head())


## Entregable

Archivos generados en `OUTPUT_ROOT` (por defecto `REPO_ROOT/colab_outputs`):

- `submission.csv`
- `rsna_knee_baseline.joblib`

Si quieres versionar salidas o checkpoints, ya quedan dentro del repositorio clonado en Colab y puedes hacer commit/push manualmente desde tu sesión.


In [ ]:
from google.colab import files

# Definir la ruta del archivo de entrega
submission_file = OUTPUT_ROOT / "submission.csv"

# Descargar el archivo directamente a tu máquina local
if submission_file.exists():
    print(f"Descargando {submission_file.name}...")
    files.download(str(submission_file))
else:
    print(f"El archivo {submission_file} no se encontró. Asegúrate de ejecutar las celdas anteriores primero.")

In [ ]:
import subprocess
from pathlib import Path

# Ruta al archivo de salida
submission_file = OUTPUT_ROOT / "submission.csv"

if submission_file.exists():
    print(f"Enviando {submission_file.name} a la competencia RSNA Knee Abnormality Detection...")
    try:
        # Ejecutar el comando de envío oficial de Kaggle capturando salidas completas
        cmd = [
            "kaggle", "competitions", "submit",
            "-c", "rsna-knee-abnormality-detection",
            "-f", str(submission_file),
            "-m", "Submission automatica desde Google Colab Baseline"
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print("Resultado del envío (stdout):")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("Error al realizar el envío (Command failed):")
        print("STDOUT:")
        print(e.stdout)
        print("STDERR:")
        print(e.stderr)
else:
    print(f"El archivo {submission_file} no existe. Por favor ejecuta el pipeline completo primero para generarlo.")